# Fraud Detection Model Fintech MLOps Stack
## Notebook 2 Preprocessing and Feature Engineering

Stage 3 takes the cleaned dataset from Notebook 1 and prepares it for modelling scaling features, splitting into train and test sets, and applying SMOTE to fix the class imbalance without data leakage.

Stage 4 enhances the raw features before they reach the model, applying log transformation to Amount, cyclical sine/cosine encoding to Time, and creating interaction features all inside a reproducible sklearn Pipeline so every transformation applied during training is automatically applied at inference time in the API.

By the end of this notebook, we have a fully preprocessed training set ready to feed into the stacking ensemble in Notebook 3.

#### Note: This notebook covers Stages 3 and 4 of our fraud detection pipeline(check notebook 1 for reference of full pipeline).

## Imports and Setup

#### We import all libraries needed for preprocessing and feature engineering. We also reload the cleaned dataset saved from Notebook 1 EDA, specifically the version after duplicate removal.

In [1]:
#  Core libraries 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter


#  Scikit-learn preprocessing 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer

#  Imbalance handling 
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

#  Serialisation 
import joblib
import os

#  Display settings 
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

print("All libraries imported successfully.")

All libraries imported successfully.


## Preprocessing Pipeline


### Load and Prepare the Cleaned Dataset

#### We reload the dataset and reapply the duplicate removal step from Notebook 1. This keeps Notebook 2 fully self-contained it can be run independently without depending on any in-memory state from Notebook 1.

In [2]:
# Load dataset 
df = pd.read_csv('../data/creditcard.csv')

# Reapply duplicate removal 
before = len(df)
df = df.drop_duplicates()
after = len(df)

print(f"Rows before duplicate removal: {before:,}")
print(f"Rows after duplicate removal:  {after:,}")
print(f"Duplicates removed:            {before - after:,}")
print()
print(f"Final dataset shape: {df.shape}")
print()
print("Class distribution:")
print(df['Class'].value_counts())

Rows before duplicate removal: 284,807
Rows after duplicate removal:  283,726
Duplicates removed:            1,081

Final dataset shape: (283726, 31)

Class distribution:
Class
0    283253
1       473
Name: count, dtype: int64


# Feature Engineering: Time & Amount (Stage 4)

## Before scaling, we need to transform Time and Amount into formats the model can actually use.
#### Time: A raw number of seconds is useless. We’ll use Sine/Cosine encoding to capture the "24-hour heartbeat."
#### Amount: Since it's heavily skewed, a Log Transformation makes the distribution more "Normal" (Gaussian), which helps the model converge faster.


In [3]:
# Check if raw columns exist before transforming
if 'Time' in df.columns and 'Amount' in df.columns:
    # 1. Convert Time to Hour
    df['hour'] = (df['Time'] / 3600) % 24
    
    # 2. Cyclical transformations
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    
    # 3. Log Transformation
    df['amount_log'] = np.log1p(df['Amount'])
    
    df_verification = df[['Time', 'hour', 'hour_sin', 'hour_cos', 'Amount', 'amount_log']].head(5)
    print("Feature Engineering Verification")
    display(df_verification)   
else:
    print("Raw 'Time', 'Hour' or 'Amount' columns not found. They might have already been dropped.")

# Verification display (only works if columns exist)
if 'hour_sin' in df.columns and 'hour' not in df.columns:
    display(df[['hour_sin', 'hour_cos', 'amount_log']].head())


Feature Engineering Verification


,Time,hour,hour_sin,hour_cos,Amount,amount_log
0,0.0000,0.0000,0.0000,1.0000,149.6200,5.0148
1,0.0000,0.0000,0.0000,1.0000,2.6900,1.3056
2,1.0000,0.0003,0.0001,1.0000,378.6600,5.9393
3,1.0000,0.0003,0.0001,1.0000,123.5000,4.8243
4,2.0000,0.0006,0.0001,1.0000,69.9900,4.2625


## Verification Analysis &  Final Column Cleanup

#### Time Check: Observe the hour_sin and hour_cos values. As hour approaches 12 (midday), sin peaks and cos hits -1. At 0 or 24 (midnight), sin is 0 and cos is 1. This confirms the circle is complete.
#### Amount Check: Compare Amount vs amount_log. An amount of $25,691 is now compressed to approximately 10.15, while $1.00 is roughly 0.69. This prevents the model from being overwhelmed by large transaction values.


In [4]:
# Dropping the original unscaled/unencoded features to prepare for Stage 3 (Splitting)
# The errors='ignore' ensures the cell runs even if columns are already gone
df = df.drop(['Time', 'Amount', 'hour'], axis=1, errors='ignore')

print(f"Final feature count: {len(df.columns)}")
print("Current columns available:", df.columns.tolist())


Final feature count: 32
Current columns available: ['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Class', 'hour_sin', 'hour_cos', 'amount_log']


##  Train/Test Split (Data Leakage Protection)

#### This is the most critical structural step in our MLOps pipeline. To ensure our model's evaluation is honest and reflects real-world performance, we must split the data before applying any scaling or oversampling (SMOTE).


In [5]:
from sklearn.model_selection import train_test_split

# 1. Define Features (X) and Target (y)
X = df.drop('Class', axis=1)
y = df['Class']

# 2. Perform Stratified Train-Test Split (80% Train, 20% Test)
# random_state=42 ensures reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Verification of Split Integrity
print(" Split Verification ")
print(f"Total Transactions: {len(df)}")
print(f"Training Set: {len(X_train)} rows")
print(f"Testing Set:  {len(X_test)} rows")

# 4. Check for Class Imbalance Consistency
train_fraud_pct = (y_train.sum() / len(y_train)) * 100
test_fraud_pct = (y_test.sum() / len(y_test)) * 100

print(f"\nFraud Rate in Training: {train_fraud_pct:.4f}%")
print(f"Fraud Rate in Testing:  {test_fraud_pct:.4f}%")


 Split Verification 
Total Transactions: 283726
Training Set: 226980 rows
Testing Set:  56746 rows

Fraud Rate in Training: 0.1665%
Fraud Rate in Testing:  0.1674%


## Feature Scaling (RobustScaler)

### Now that the split is done, we need to scale the features. Remember our EDA? Some features were small, but amount_log is on a different scale.

#### Why RobustScaler? Unlike StandardScaler (which uses the Mean), RobustScaler uses the Median and Interquartile Range (IQR).

#### The Benefit: It is "robust" to outliers. Since fraud data is defined by outliers, this scaler ensures that extreme values don't "stretch" our feature space too much, keeping the math stable for our model.


In [6]:
# 1. Initialize the Scaler
scaler = RobustScaler()

# 2. FIT and TRANSFORM the Training data
# "Fitting" means the scaler learns the median/IQR of the training set ONLY
X_train_scaled = scaler.fit_transform(X_train)

# 3. TRANSFORM the Test data
# We ONLY transform here. We never "fit" on test data (that would be leakage!)
X_test_scaled = scaler.transform(X_test)

# 4. Convert back to DataFrame for readability (optional but helpful)
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print(" Scaling Verification ")
print(f"X_train Max (V17): {X_train['V17'].max():.2f}")
print(f"X_train Min (V17): {X_train['V17'].min():.2f}")
display(X_train.head(3))

 Scaling Verification 
X_train Max (V17): 10.58
X_train Min (V17): -28.49


,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,hour_sin,hour_cos,amount_log
0,0.9952,-1.2752,-1.2168,-1.6067,0.8019,3.2941,-1.6247,1.5688,-0.9779,1.6782,-0.1361,-0.8259,0.2875,-0.3944,0.1917,-0.6523,0.5076,-0.1796,-0.3530,-0.7582,-0.2912,-0.0530,0.9333,0.8097,-0.3516,-0.1897,0.2249,-0.4673,-0.4136,-0.1059,0.1449
1,-0.5976,1.1173,0.2187,-0.0103,-0.2699,-0.5915,0.1525,1.3352,-0.9811,-0.3929,0.9779,1.5902,0.9674,0.8857,-0.2564,0.3286,-0.2267,-0.3423,0.4235,-0.0157,-0.5064,-0.8913,1.0797,0.5969,-1.0901,-0.3791,-0.1191,0.3014,0.0308,0.8564,-0.4300
2,0.8472,-0.0304,-1.1836,0.0951,0.9306,0.7556,-0.0562,0.2959,0.2718,-0.4082,1.0417,0.8822,0.3731,-0.8036,0.7437,-0.3493,0.9751,-0.1555,-1.0923,-0.4298,0.7766,1.0167,-0.1078,-2.1771,0.1344,0.1049,0.2697,-0.4911,0.5650,-0.4674,-0.1498


## SMOTE(Synthetic minority oversampling technique): Balancing the Playing Field

#### We have arrived at the most critical "Fintech" adjustment in our pipeline. Our training set is currently a "ghost town" for fraud examples—only 394 fraudulent cases against 227,000 legitimate ones.

#### The Problem: If we train now, the model will develop a "Majority Bias." It will realize it can achieve 99.9% accuracy simply by predicting "Legitimate" every single time, effectively ignoring the fraud signal.

#### The Solution (SMOTE): Synthetic Minority Over-sampling Technique. Instead of just making copies of the fraud rows (which leads to overfitting), SMOTE selects a fraud data point, finds its "nearest neighbors," and creates entirely new, synthetic transactions that fall between them.

### Safety Rule: We only apply this to the training set. The test set must remain "unbalanced" so we can see how the model performs in the real world

In [7]:
# 1. Initialize SMOTE
# sampling_strategy=1.0 ensures the fraud class ends up with the same count as legitimate
smote = SMOTE(sampling_strategy=1.0, random_state=42)

# 2. Resample the training data
print(f"Before SMOTE (Training set): {Counter(y_train)}")

X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# 3. Verify the new balanced state
print(f"After SMOTE (Training set):  {Counter(y_train_res)}")

# 4. Final check of the shapes
print(f"\nFinal Preprocessed Training Shape: {X_train_res.shape}")
print(f"Final Preprocessed Testing Shape:  {X_test.shape}")

Before SMOTE (Training set): Counter({0: 226602, 1: 378})
After SMOTE (Training set):  Counter({0: 226602, 1: 226602})

Final Preprocessed Training Shape: (453204, 31)
Final Preprocessed Testing Shape:  (56746, 31)
